# 04 - Knowledge Distillation: Baseline Teacher -> Small Student

Phase 4 of the end-to-end pipeline (train -> evaluate -> video inference -> **distill** ->
ONNX/TensorRT -> Jetson Orin Nano). The baseline from `01_training.ipynb` is compressed
into a smaller student that keeps as much of its accuracy as possible:

```
models/baseline/best.pt  (EfficientNet-B0 teacher, frozen)
        |
        |  soft targets: softmax(teacher logits / T)   <- "dark knowledge": how the teacher
        |                                                 spreads probability over the
        |                                                 196 visually similar car classes
        v
MobileNetV3-Small student (ImageNet-pretrained, ~2.5M parameters)
        |
        v
models/distilled/best.pt   <- same checkpoint schema as 01, ready for ONNX/TensorRT export
```

Training loss (classic Hinton-style distillation):

```
L = ALPHA * T^2 * KL( softmax(teacher/T) || log_softmax(student/T) )   soft-target loss
  + (1 - ALPHA) * CE( student logits, labels )                         ordinary hard-label loss
```

- **Temperature T > 1** softens both distributions so the inter-class similarity structure
  survives the softmax (at T = 1 the teacher is usually near one-hot).
- The **T^2 factor** keeps gradient magnitudes comparable across temperatures.
- **ALPHA** balances soft vs. hard targets.
- The student is always evaluated **alone** - the teacher exists only during training.

Notes for this demo run:

- The student starts from **ImageNet** weights only. No third-party Stanford-Cars
  pretrained checkpoints are used anywhere; the teacher is the baseline we trained in 01.
- The notebook runs **top to bottom in Google Colab**. Recommended: GPU runtime
  (Runtime -> Change runtime type -> T4 GPU).
- Needed before running: the dataset at `/content/data/stanford_cars` (section 3 shows how
  to get it there) and the teacher `models/baseline/best.pt` from 01 (run 01 first in the
  same Colab session, or upload the file).

## 1. Environment

In [ ]:
import os, sys, time, random, textwrap

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torchvision
from PIL import Image
from torch.utils.data import DataLoader, Dataset, Subset

%matplotlib inline

print("Python        :", sys.version.split()[0])
print("PyTorch       :", torch.__version__)
print("torchvision   :", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU           :", torch.cuda.get_device_name(0))

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device        :", DEVICE, "(seed =", SEED, ")")
if DEVICE.type == "cpu":
    print("NOTE: running on CPU - in Colab select a GPU runtime for reasonable speed.")

## 2. Configuration

Paths default to the Colab filesystem (`/content`) when running there and fall back to
relative repo paths locally. All can be overridden with environment variables.

In [ ]:
from pathlib import Path

IN_COLAB = os.path.isdir("/content")   # running in Google Colab?

if IN_COLAB:
    _DATA_DEFAULT, _MODELS_DEFAULT = "/content/data/stanford_cars", "/content/models"
else:
    _DATA_DEFAULT, _MODELS_DEFAULT = "../data/stanford_cars", "../models"

DATASET_ROOT = Path(os.environ.get("STANFORD_CARS_ROOT", _DATA_DEFAULT))
MODELS_DIR   = Path(os.environ.get("MODELS_DIR", _MODELS_DEFAULT))
TEACHER_CKPT = Path(os.environ.get("TEACHER_CKPT", str(MODELS_DIR / "baseline" / "best.pt")))
STUDENT_CKPT = Path(os.environ.get("STUDENT_CKPT", str(MODELS_DIR / "distilled" / "best.pt")))

IMAGE_SIZE    = 224     # must match the teacher checkpoint (asserted in section 4)
BATCH_SIZE    = 32
NUM_EPOCHS    = 10
LEARNING_RATE = 1e-3
WEIGHT_DECAY  = 1e-4
NUM_WORKERS   = 2
VAL_FRACTION  = 0.1     # must match 01_training.ipynb so the split (and metrics) line up
SEED          = 42

TEMPERATURE = 4.0       # KD temperature
ALPHA       = 0.7       # weight of the soft-target loss; hard-label loss gets (1 - ALPHA)

print("running in Colab:", IN_COLAB)
print("DATASET_ROOT    :", DATASET_ROOT)
print("TEACHER_CKPT    :", TEACHER_CKPT)
print("STUDENT_CKPT    :", STUDENT_CKPT)
print("NUM_EPOCHS      :", NUM_EPOCHS)
print("TEMPERATURE     :", TEMPERATURE)
print("ALPHA           :", ALPHA, "(soft weight | hard weight =", 1 - ALPHA, ")")

## 3. Dataset (Stanford Cars)

Get the dataset into the Colab filesystem at `/content/data/stanford_cars` **once per
session** before running the rest. Two common options - run them in a scratch cell:

**Option A - Kaggle mirror** (needs a free Kaggle account and its API token
`kaggle.json`; the mirror below unzips to the class-folder layout used here):

    from google.colab import files; files.upload()   # upload kaggle.json
    !pip -q install kaggle
    !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
    !kaggle datasets download -d jutrera/stanford-car-dataset-by-classes-folder -p /content/data
    !mkdir -p /content/data/stanford_cars
    !unzip -q /content/data/stanford-car-dataset-by-classes-folder.zip -d /content/data/stanford_cars

**Option B - Google Drive** (upload the dataset to Drive once, then mount it):

    from google.colab import drive; drive.mount('/content/drive')
    # then set DATASET_ROOT = Path('/content/drive/MyDrive/<your dataset folder>') above

Two layouts are supported (auto-detected):

**Layout A - official devkit files** (needs `scipy`): `cars_meta.mat`, `cars_annos.mat`,
`cars_train/`, `cars_test/`.

**Layout B - one folder per class**: `train/<class name>/*.jpg`, `test/<class name>/*.jpg`
(what the Kaggle mirror above provides).

Stanford Cars: 16,185 images, 196 classes (8,144 train / 8,041 test). The dataset is
**not** downloaded automatically by the notebook - original URLs are unreliable.

In [ ]:
def discover_stanford_cars(root):
    """Find a Stanford Cars layout under `root`.

    Returns {"classes": [str], "train": [(path, label)], "test": [(path, label)]}
    """
    root = Path(root)
    if not root.exists():
        raise FileNotFoundError(
            "DATASET_ROOT does not exist: " + str(root)
            + "\nSet the STANFORD_CARS_ROOT environment variable or edit the configuration cell.")
    if (root / "cars_meta.mat").exists() and (root / "cars_annos.mat").exists():
        return _discover_devkit(root)
    if (root / "train").is_dir() and (root / "test").is_dir():
        return _discover_class_folders(root)
    raise FileNotFoundError(
        "No supported Stanford Cars layout under " + str(root) + "\n"
        + "Layout A (devkit):  cars_meta.mat, cars_annos.mat, cars_train/, cars_test/\n"
        + "Layout B (folders): train/<class name>/*.jpg, test/<class name>/*.jpg\n"
        + "Download the dataset manually (e.g. the Kaggle mirror "
        + "jutrera/stanford-car-dataset-by-classes-folder).")


def _discover_devkit(root):
    from scipy.io import loadmat  # only needed for this layout
    class_names = [str(c[0]) for c in loadmat(root / "cars_meta.mat")["class_names"].ravel()]
    train_samples, test_samples = [], []
    for a in loadmat(root / "cars_annos.mat")["annotations"].ravel():
        rel     = str(np.asarray(a["relative_im_path"]).ravel()[0])
        label   = int(np.asarray(a["class"]).ravel()[0]) - 1
        is_test = bool(int(np.asarray(a["test"]).ravel()[0]))
        (test_samples if is_test else train_samples).append((root / rel, label))
    return {"classes": class_names, "train": train_samples, "test": test_samples}


def _discover_class_folders(root):
    classes = sorted(p.name for p in (root / "train").iterdir() if p.is_dir())
    class_to_idx = {name: i for i, name in enumerate(classes)}
    result = {"classes": classes, "train": [], "test": []}
    for split in ("train", "test"):
        for class_dir in sorted((root / split).iterdir()):
            if not class_dir.is_dir():
                continue
            label = class_to_idx[class_dir.name]
            for pattern in ("*.jpg", "*.jpeg", "*.png"):
                for img_path in sorted(class_dir.glob(pattern)):
                    result[split].append((img_path, label))
    return result


class StanfordCarsDataset(Dataset):
    """Dataset over (path, label) pairs; transform is passed in so train and
    eval copies can use different preprocessing over the same images."""

    def __init__(self, samples, classes, transform=None):
        self.samples = samples
        self.classes = classes
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        path, label = self.samples[index]
        image = Image.open(path).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        return image, label

In [ ]:
data = discover_stanford_cars(DATASET_ROOT)
classes       = data["classes"]
train_samples = data["train"]
test_samples  = data["test"]

assert len(classes) == 196, "expected 196 classes, found %d" % len(classes)
print("train samples:", len(train_samples))
print("test samples :", len(test_samples))
print("num classes  :", len(classes))
print("example class:", classes[0])

## 4. Preprocessing (inherited from the teacher)

The student must see **exactly** the inputs the teacher saw, so the preprocessing
dictionary is taken from the teacher checkpoint rather than redefined here - one source of
truth, same as promised in 01.

In [ ]:
from torchvision import transforms

MEAN = [0.485, 0.456, 0.406]   # ImageNet statistics
STD  = [0.229, 0.224, 0.225]

PREPROCESSING = {
    "image_size": IMAGE_SIZE,
    "mean": MEAN,
    "std": STD,
    "interpolation": "bilinear",
}

def build_transforms(preprocessing, augment):
    """Build the transform for one mode from a PREPROCESSING dict.

    augment=True  -> training: random crop + horizontal flip
    augment=False -> eval / inference: deterministic resize
    """
    size = preprocessing["image_size"]
    if augment:
        spatial = [transforms.RandomResizedCrop(size, scale=(0.6, 1.0)),
                   transforms.RandomHorizontalFlip()]
    else:
        spatial = [transforms.Resize((size, size))]
    return transforms.Compose(
        spatial
        + [transforms.ToTensor(),
           transforms.Normalize(preprocessing["mean"], preprocessing["std"])])

train_tf = build_transforms(PREPROCESSING, augment=True)
eval_tf  = build_transforms(PREPROCESSING, augment=False)
print("train transform:", train_tf)
print()
print("eval transform :", eval_tf)

In [ ]:
assert TEACHER_CKPT.exists(), (
    "teacher checkpoint not found: " + str(TEACHER_CKPT)
    + "\nRun 01_training.ipynb first in this Colab session (it saves models/baseline/best.pt)"
    + " or upload the file to that path.")

teacher_ckpt = torch.load(TEACHER_CKPT, map_location="cpu", weights_only=False)  # our own checkpoint

assert teacher_ckpt["preprocessing"]["image_size"] == IMAGE_SIZE, (
    "IMAGE_SIZE does not match the teacher checkpoint - "
    "set IMAGE_SIZE to %d" % teacher_ckpt["preprocessing"]["image_size"])

PREPROCESSING = teacher_ckpt["preprocessing"]
train_tf = build_transforms(PREPROCESSING, augment=True)
eval_tf  = build_transforms(PREPROCESSING, augment=False)

print("teacher        :", TEACHER_CKPT)
print("architecture   :", teacher_ckpt["model_arch"])
print("preprocessing  :", PREPROCESSING)
print("teacher metrics:", teacher_ckpt["metrics"])

## 5. Teacher and student

- **Teacher** - rebuilt from the baseline checkpoint, frozen (`eval()` + no gradients);
  it only produces logits to imitate.
- **Student** - **MobileNetV3-Small** (~2.5M parameters, roughly half of EfficientNet-B0),
  an architecture designed for mobile/edge deployment and a realistic stand-in for what
  will be exported to ONNX/TensorRT and run on the Jetson Orin Nano. Its 1000-class
  ImageNet head is replaced with the 196-class head.

In [ ]:
STUDENT_ARCH = "mobilenet_v3_small"

def build_model_from_checkpoint(ckpt):
    """Rebuild any architecture recorded in a project checkpoint (baseline or student)."""
    arch = ckpt["model_arch"]
    if arch == "efficientnet_b0":
        from torchvision.models import efficientnet_b0
        model = efficientnet_b0(weights=None)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, ckpt["num_classes"])
    elif arch == "mobilenet_v3_small":
        from torchvision.models import mobilenet_v3_small
        model = mobilenet_v3_small(weights=None)
        model.classifier[3] = nn.Linear(model.classifier[3].in_features, ckpt["num_classes"])
    else:
        raise ValueError("unknown architecture: " + str(arch))
    return model


def build_student(num_classes, pretrained=True):
    from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights
    weights = MobileNet_V3_Small_Weights.IMAGENET1K_V1 if pretrained else None
    model = mobilenet_v3_small(weights=weights)
    model.classifier[3] = nn.Linear(model.classifier[3].in_features, num_classes)
    return model


teacher = build_model_from_checkpoint(teacher_ckpt).to(DEVICE).eval()
for param in teacher.parameters():
    param.requires_grad_(False)

student = build_student(teacher_ckpt["num_classes"], pretrained=True).to(DEVICE)


def param_count(m):
    return sum(p.numel() for p in m.parameters())


print("teacher : efficientnet_b0      %6.2f M params | val top-1 %.2f%%"
      % (param_count(teacher) / 1e6, teacher_ckpt["metrics"]["val_top1"]))
print("student : mobilenet_v3_small  %6.2f M params | %.0f%% of the teacher"
      % (param_count(student) / 1e6, param_count(student) / param_count(teacher) * 100))

## 6. Distillation training

Per batch, both networks see the same images:

```
images --> teacher (frozen) --> teacher_logits --+
                                                  +--> soft loss: T^2 * KL  (weight ALPHA)
images --> student ----------------> student_logits -+
                                                  |
labels --------------------------------------------+--> hard loss: CE  (weight 1 - ALPHA)
```

The validation pass uses the student only, with the same `accuracy` / `evaluate` helpers
as 01 - comparable numbers, no teacher involvement. The train/val split reproduces 01's
exactly (same seed and fraction).

In [ ]:
# Identical split logic to 01 (same SEED and VAL_FRACTION) -> identical train/val sets.
val_len = int(len(train_samples) * VAL_FRACTION)
perm = list(range(len(train_samples)))
random.Random(SEED).shuffle(perm)
val_indices, train_indices = perm[:val_len], perm[val_len:]

train_ds = StanfordCarsDataset(train_samples, classes, transform=train_tf)
val_ds   = StanfordCarsDataset(train_samples, classes, transform=eval_tf)
test_ds  = StanfordCarsDataset(test_samples,  classes, transform=eval_tf)

pin = DEVICE.type == "cuda"
train_loader = DataLoader(Subset(train_ds, train_indices), batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=pin)
val_loader   = DataLoader(Subset(val_ds, val_indices), batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=pin)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(student.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

print("train/val split: %d / %d (same as 01: seed=%d, VAL_FRACTION=%s)"
      % (len(train_indices), len(val_indices), SEED, VAL_FRACTION))


def kd_loss(student_logits, teacher_logits, targets, temperature, alpha):
    """Hinton-style distillation loss. Returns (total, soft, hard) for logging."""
    soft = nn.functional.kl_div(
        nn.functional.log_softmax(student_logits / temperature, dim=1),
        nn.functional.softmax(teacher_logits / temperature, dim=1),
        reduction="batchmean",
    ) * (temperature ** 2)
    hard = nn.functional.cross_entropy(student_logits, targets)
    total = alpha * soft + (1.0 - alpha) * hard
    return total, soft.detach(), hard.detach()


def accuracy(outputs, targets, topk=(1,)):
    """Per-sample Top-k accuracy in percent for each k in topk."""
    maxk = max(topk)
    _, pred = outputs.topk(maxk, dim=1, largest=True, sorted=True)
    correct = pred.t().eq(targets[None])
    return [correct[:k].any(dim=0).float().mean().item() * 100.0 for k in topk]


def train_one_epoch(student, teacher, loader, optimizer, temperature, alpha):
    student.train()
    total, soft_sum, hard_sum, n = 0.0, 0.0, 0.0, 0
    for images, targets in loader:
        images, targets = images.to(DEVICE), targets.to(DEVICE)
        with torch.no_grad():
            teacher_logits = teacher(images)
        student_logits = student(images)
        loss, soft_l, hard_l = kd_loss(student_logits, teacher_logits, targets, temperature, alpha)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        bs = targets.size(0)
        total += loss.item() * bs
        soft_sum += soft_l.item() * bs
        hard_sum += hard_l.item() * bs
        n += bs
    return total / n, soft_sum / n, hard_sum / n


@torch.no_grad()
def evaluate(model, loader, criterion):
    """Return (mean loss, Top-1 %, Top-5 %) over the loader - student only."""
    model.eval()
    total_loss, top1, top5, n = 0.0, 0.0, 0.0, 0
    for images, targets in loader:
        images, targets = images.to(DEVICE), targets.to(DEVICE)
        outputs = model(images)
        total_loss += criterion(outputs, targets).item() * targets.size(0)
        a1, a5 = accuracy(outputs, targets, topk=(1, 5))
        top1 += a1 * targets.size(0)
        top5 += a5 * targets.size(0)
        n += targets.size(0)
    return total_loss / n, top1 / n, top5 / n

In [ ]:
history = {"train_loss": [], "soft_loss": [], "hard_loss": [],
           "val_loss": [], "val_top1": [], "val_top5": [], "lr": []}
best = {"val_top1": -1.0, "val_top5": -1.0, "epoch": -1, "state_dict": None}

for epoch in range(1, NUM_EPOCHS + 1):
    t0 = time.time()
    train_loss, soft_loss, hard_loss = train_one_epoch(
        student, teacher, train_loader, optimizer, TEMPERATURE, ALPHA)
    val_loss, val_top1, val_top5 = evaluate(student, val_loader, criterion)

    history["train_loss"].append(train_loss)
    history["soft_loss"].append(soft_loss)
    history["hard_loss"].append(hard_loss)
    history["val_loss"].append(val_loss)
    history["val_top1"].append(val_top1)
    history["val_top5"].append(val_top5)
    history["lr"].append(optimizer.param_groups[0]["lr"])

    marker = ""
    if val_top1 > best["val_top1"]:
        best = {
            "val_top1": val_top1,
            "val_top5": val_top5,
            "epoch": epoch,
            "state_dict": {k: v.detach().cpu().clone() for k, v in student.state_dict().items()},
        }
        marker = "  <- best"

    scheduler.step()
    print("epoch %2d/%d | loss %.4f (soft %.3f + hard %.3f) | val loss %.4f | top1 %5.2f%% | top5 %5.2f%% | %5.1fs%s"
          % (epoch, NUM_EPOCHS, train_loss, soft_loss, hard_loss, val_loss, val_top1, val_top5,
             time.time() - t0, marker))

## 7. Training curves

The soft loss usually drops much faster than the hard loss - the student first learns the
teacher's *relative* class structure, then sharpens its own decisions.

In [ ]:
fig, (ax_loss, ax_acc) = plt.subplots(1, 2, figsize=(13, 4))
epochs = range(1, NUM_EPOCHS + 1)

ax_loss.plot(epochs, history["train_loss"], marker="o", label="total (ALPHA*soft + (1-ALPHA)*hard)")
ax_loss.plot(epochs, history["soft_loss"], marker="o", label="soft loss (KL, T-scaled)")
ax_loss.plot(epochs, history["hard_loss"], marker="o", label="hard loss (CE)")
ax_loss.plot(epochs, history["val_loss"], marker="o", linestyle="--", label="val loss")
ax_loss.set_xlabel("epoch"); ax_loss.set_ylabel("loss")
ax_loss.set_title("Distillation loss components"); ax_loss.legend(); ax_loss.grid(alpha=0.3)

ax_acc.plot(epochs, history["val_top1"], marker="o", label="val top-1")
ax_acc.plot(epochs, history["val_top5"], marker="o", label="val top-5")
ax_acc.axvline(best["epoch"], color="gray", linestyle="--", label="best epoch")
ax_acc.set_xlabel("epoch"); ax_acc.set_ylabel("accuracy (%)")
ax_acc.set_title("Student validation accuracy"); ax_acc.legend(); ax_acc.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Save checkpoint

The **best** student (by validation top-1) is saved in the **same schema as 01** -
architecture, weights, class mapping, preprocessing, config, metrics, history - so the
same loader code that reads the baseline also reads the student. `training_config`
additionally records the distillation hyperparameters and the teacher it came from. This
checkpoint is the artifact for the next phase: ONNX -> TensorRT -> Jetson Orin Nano.

In [ ]:
checkpoint = {
    "format_version": 1,
    "dataset": "stanford_cars",
    "model_arch": STUDENT_ARCH,
    "state_dict": best["state_dict"],
    "classes": classes,
    "num_classes": len(classes),
    "preprocessing": PREPROCESSING,
    "training_config": {
        "image_size": IMAGE_SIZE,
        "batch_size": BATCH_SIZE,
        "num_epochs": NUM_EPOCHS,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "val_fraction": VAL_FRACTION,
        "seed": SEED,
        "optimizer": "adamw",
        "scheduler": "cosine",
        "distillation": True,
        "temperature": TEMPERATURE,
        "alpha": ALPHA,
        "teacher_checkpoint": str(TEACHER_CKPT),
        "teacher_arch": teacher_ckpt["model_arch"],
    },
    "metrics": {
        "val_top1": best["val_top1"],
        "val_top5": best["val_top5"],
        "val_loss": history["val_loss"][best["epoch"] - 1],
        "epoch": best["epoch"],
        "teacher_val_top1": teacher_ckpt["metrics"]["val_top1"],
    },
    "history": history,
    "torch_version": str(torch.__version__),
    "saved_at": time.strftime("%Y-%m-%d %H:%M:%S"),
}

STUDENT_CKPT.parent.mkdir(parents=True, exist_ok=True)
torch.save(checkpoint, STUDENT_CKPT)
print("saved      :", STUDENT_CKPT, "(%.1f MB)" % (STUDENT_CKPT.stat().st_size / 1e6))
print("best epoch :", best["epoch"])
print("val top-1  : %.2f%%" % best["val_top1"])
print("val top-5  : %.2f%%" % best["val_top5"])
if IN_COLAB:
    print("\nColab tip: download the checkpoint now (Files sidebar or files.download) -")
    print("the /content filesystem is wiped when the session ends.")

## 9. Sanity-check inference

Reload the student checkpoint **from disk** the way the export notebook will and run it on
a few test images - a quick check that the artifact is self-contained, not a real
evaluation.

In [ ]:
loaded = torch.load(STUDENT_CKPT, map_location=DEVICE, weights_only=False)  # our own checkpoint

student_model = build_model_from_checkpoint(loaded).to(DEVICE)
student_model.load_state_dict(loaded["state_dict"])
student_model.eval()

test_ds_loaded = StanfordCarsDataset(test_samples, loaded["classes"],
                                     transform=build_transforms(loaded["preprocessing"], augment=False))


def short_name(name):
    return "\n".join(textwrap.wrap(name, 24))


picks = random.Random(SEED).sample(range(len(test_ds_loaded)), 8)
inv_mean = torch.tensor(loaded["preprocessing"]["mean"])[:, None, None]
inv_std  = torch.tensor(loaded["preprocessing"]["std"])[:, None, None]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, i in zip(axes.ravel(), picks):
    image, target = test_ds_loaded[i]
    with torch.no_grad():
        probs = torch.softmax(student_model(image[None].to(DEVICE)), dim=1)[0].cpu()
    conf, pred = probs.max(0)
    ok = pred.item() == target
    show = (image * inv_std + inv_mean).clamp(0, 1).permute(1, 2, 0).numpy()
    ax.imshow(show)
    title = ("OK  " if ok else "WRONG\n") + "gt:   " + short_name(loaded["classes"][target])
    title += "\npred: " + short_name(loaded["classes"][pred.item()]) + " (%.1f%%)" % (conf.item() * 100)
    ax.set_title(title, fontsize=8, color="green" if ok else "red")
    ax.axis("off")
plt.suptitle("Student sanity check (checkpoint loaded from disk)", y=1.02)
plt.tight_layout()
plt.show()

## 10. Teacher vs. student at a glance

Validation metrics come from the **same split** (same seed as 01), and latency uses the
same single-image protocol as `02_evaluation.ipynb`. This demo is not a research-grade
benchmark - the point is to see the compression trade-off the pipeline will carry onto
the Jetson.

In [ ]:
def measure_latency(model, device, iters, warmup, image_size):
    """Mean forward-pass time per single image, in milliseconds."""
    x = torch.randn(1, 3, image_size, image_size, device=device)
    with torch.no_grad():
        for _ in range(warmup):
            model(x)
        if device.type == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        for _ in range(iters):
            model(x)
        if device.type == "cuda":
            torch.cuda.synchronize()
    return (time.perf_counter() - t0) / iters * 1000.0


lat_teacher = measure_latency(teacher, DEVICE, 200, 20, PREPROCESSING["image_size"])
lat_student = measure_latency(student, DEVICE, 200, 20, PREPROCESSING["image_size"])

rows = [
    ("architecture",     teacher_ckpt["model_arch"], STUDENT_ARCH),
    ("parameters (M)",   "%.2f" % (param_count(teacher) / 1e6), "%.2f" % (param_count(student) / 1e6)),
    ("weights fp32 (MB)","%.1f" % (param_count(teacher) * 4 / 1e6), "%.1f" % (param_count(student) * 4 / 1e6)),
    ("checkpoint (MB)",  "%.1f" % (TEACHER_CKPT.stat().st_size / 1e6), "%.1f" % (STUDENT_CKPT.stat().st_size / 1e6)),
    ("val top-1 (%)",    "%.2f" % teacher_ckpt["metrics"]["val_top1"], "%.2f" % best["val_top1"]),
    ("val top-5 (%)",    "%.2f" % teacher_ckpt["metrics"]["val_top5"], "%.2f" % best["val_top5"]),
    ("latency ms/img",   "%.2f" % lat_teacher, "%.2f" % lat_student),
    ("throughput fps",   "%.0f" % (1000.0 / lat_teacher), "%.0f" % (1000.0 / lat_student)),
]

print("%-20s | %-22s | %-22s" % ("", "teacher", "student"))
print("%-20s-+-%-22s-+-%-22s" % ("-" * 20, "-" * 22, "-" * 22))
for name, t, s in rows:
    print("%-20s | %-22s | %-22s" % (name, str(t), str(s)))

print()
print("student = %.0f%% of teacher params, %.2fx latency, val top-1 gap %.2f points"
      % (param_count(student) / param_count(teacher) * 100,
         lat_teacher / lat_student,
         teacher_ckpt["metrics"]["val_top1"] - best["val_top1"]))

## Summary

- Distilled the EfficientNet-B0 baseline into a **MobileNetV3-Small** student with
  classic soft-target (temperature, alpha) + hard-label loss - 10 epochs, best checkpoint
  by validation top-1 saved to `models/distilled/best.pt`.
- The checkpoint uses the **same schema as the baseline**, so the evaluation and video
  notebooks load it with a single path change.
- The student starts from ImageNet weights only; no third-party Stanford-Cars checkpoint
  is involved anywhere.

Next phase (deployment):

1. Export the student to **ONNX** (fixed 1x3x224x224 input).
2. Build a **TensorRT** engine on the Jetson Orin Nano.
3. Run real-time inference and benchmark FPS on the device - this notebook's compression
   table is the baseline reference for that benchmark.